<a href="https://colab.research.google.com/github/Rafiatun/Study-Impact/blob/main/UU_Student_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import json
import warnings
from math import pi

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    r2_score, mean_absolute_error,
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LassoCV
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import (
    train_test_split, cross_val_score, KFold, GridSearchCV,
)
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
plt.rcParams["font.family"] = "serif"

In [2]:
DATA_PATH = "/content/sample_data/Dataset_200_students (4).csv"
OUTDIR = "figs"
RNG = 42
K = 3                 # number of clusters
TEST_SIZE = 0.20      # 20% held out for final evaluation
CV_FOLDS = 5

os.makedirs(OUTDIR, exist_ok=True)

# Capture every console line so we can save the full log at the end.
LOG = []
def log(msg=""):
    print(msg)
    LOG.append(str(msg))

def section(title):
    log("\n" + "=" * 70)
    log(title)
    log("=" * 70)


STEP 1. LOAD AND CLEAN

In [3]:
# =============================================================================
section("STEP 1 — LOAD AND CLEAN")

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
for col in df.select_dtypes("object").columns:
    df[col] = df[col].str.strip()

log(f"Rows: {len(df)}, columns: {len(df.columns)}")
missing = df.isna().sum()
log("Missing values:")
log(missing[missing > 0].to_string() if missing.sum() else "  none")


STEP 1 — LOAD AND CLEAN
Rows: 200, columns: 21
Missing values:
work_hrs    121


# STEP 2. ENCODE VARIABLES

In [4]:
# =============================================================================
section("STEP 2 — ENCODE VARIABLES")

# 2a. Outcome: CGPA bands -> band midpoints
df["CGPA"] = df["CGPA?"].map({
    "2.0-2.5": 2.25, "2.5-3.0": 2.75, "3.0-3.5": 3.25, "3.5-4.0": 3.75,
})

# 2b. Ordinal scales (preserve natural order)
df["study_hrs_n"] = df["study_hrs"].map({
    "Less than 1 hour": 1, "1-2 hours": 2, "3-4 hours": 3, "More than 4 hours": 4,
})
df["sleep_hrs_n"] = df["sleep_hrs"].map({
    "Less than 4 hours": 1, "4-6 hours": 2, "6-8 hours": 3, "More than 8 hours": 4,
})
df["screen_n"] = df["screen_time"].map({
    "Less than 1 hour": 1, "1-3 hours": 2, "4-6 hours": 3, "More than 6 hours": 4,
})
df["travel_n"] = df["travel_time"].map({
    "Less than 15 minutes": 1, "15-30 minutes": 2, "30-60 minutes": 3,
    "More than 1 hour": 4, "More than 2 hour": 5,
})
df["stress_n"] = df["stress"].map({
    "Never": 1, "Rarely": 2, "Sometimes": 3, "Often": 4, "Always": 5,
})
df["fin_cond_n"] = df["fin_cond"].map({
    "Low Income": 1, "Middle Income": 2, "High Income": 3,
})
df["balance_n"] = df["balance_diff"].map({"No": 1, "Maybe": 2, "Yes": 3})
df["work_hrs_n"] = df["work_hrs"].map({
    "Less than 5 hours": 4, "5-10 hours": 7.5,
    "11-20 hours": 15, "More than 20 hours": 22,
}).fillna(0)

# 2c. Cyclic encoding for time of day (Morning=6, Evening=18, Late Night=24)
df["study_h"] = df["study_time"].map({"Morning": 6, "Evening": 18, "Late Night": 24})
df["time_sin"] = np.sin(2 * pi * df["study_h"] / 24)
df["time_cos"] = np.cos(2 * pi * df["study_h"] / 24)

# 2d. Yes/No -> 0/1
for col in ["pt_job", "laptop", "ECA", "travel_impact", "screen_sleep"]:
    df[col + "_b"] = df[col].map({"Yes": 1, "No": 0})
df["Gender_b"] = df["Gender"].map({"Male": 1, "Female": 0})

# 2e. Multi-label phone use -> one-hot
phone = df["phone_use"].fillna("").str.get_dummies(sep=", ").astype(int)
phone.columns = [c.strip().replace(" ", "_") for c in phone.columns]
for col in ["Study_Purpose", "Social_Media", "Entertainment", "Gaming"]:
    if col not in phone.columns:
        phone[col] = 0
df = pd.concat([df, phone], axis=1)

# 2f. Transport mode (small set, ordered by typical convenience)
df["trans_n"] = df["trans_mode"].map({
    "University Bus": 0, "Public Transport": 1, "Private Vehicle": 2,
})

log("Encoded all variables.")




STEP 2 — ENCODE VARIABLES
Encoded all variables.


# STEP 3. DESCRIPTIVES

In [5]:
# =============================================================================
section("STEP 3 — DESCRIPTIVES")

log(f"Age: mean = {df['Age'].mean():.2f}, sd = {df['Age'].std():.2f}, "
    f"range = {df['Age'].min()}–{df['Age'].max()}")
for col in ["Gender", "CGPA?", "study_hrs", "travel_time", "stress"]:
    log(f"\n{col}:\n{df[col].value_counts().to_string()}")


STEP 3 — DESCRIPTIVES
Age: mean = 22.21, sd = 1.40, range = 19–25

Gender:
Gender
Male      177
Female     23

CGPA?:
CGPA?
3.5-4.0    69
3.0-3.5    68
2.5-3.0    53
2.0-2.5    10

study_hrs:
study_hrs
1-2 hours            76
Less than 1 hour     56
3-4 hours            51
More than 4 hours    17

travel_time:
travel_time
More than 2 hour        90
More than 1 hour        44
15-30 minutes           30
30-60 minutes           26
Less than 15 minutes    10

stress:
stress
Sometimes    107
Often         45
Always        26
Rarely        20
Never          2


# STEP 4. CORRELATIONS WITH CGPA (with 95% CI)

In [6]:
# =============================================================================
section("STEP 4 — CORRELATIONS WITH CGPA")

def fisher_ci(r, n, alpha=0.05):
    """95% CI for Pearson r using Fisher z-transform."""
    if abs(r) >= 1:
        return r, r
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    zc = stats.norm.ppf(1 - alpha / 2)
    return np.tanh(z - zc * se), np.tanh(z + zc * se)

corr_vars = {
    "Academic self-rating": "acad_rating",
    "Daily study hours": "study_hrs_n",
    "Travel time": "travel_n",
    "Avoid screen before sleep": "screen_sleep_b",
    "Phone: study purpose": "Study_Purpose",
    "Time of study (morning bias)": "time_sin",
    "Phone: social media": "Social_Media",
    "Perceived stress": "stress_n",
    "Sleep duration": "sleep_hrs_n",
    "Daily screen time": "screen_n",
    "Extracurriculars": "ECA_b",
    "Part-time job": "pt_job_b",
    "Phone: entertainment": "Entertainment",
    "Weekly work hours": "work_hrs_n",
    "Travel impact (perceived)": "travel_impact_b",
    "Financial condition": "fin_cond_n",
    "Phone: gaming": "Gaming",
    "Gender (Male=1)": "Gender_b",
    "Age": "Age",
}

rows = []
for label, col in corr_vars.items():
    s = df[[col, "CGPA"]].dropna()
    r, p = stats.pearsonr(s[col], s["CGPA"])
    lo, hi = fisher_ci(r, len(s))
    rows.append({"variable": label, "r": r, "ci_lo": lo, "ci_hi": hi, "p": p})

corr_df = pd.DataFrame(rows).sort_values("r", key=abs, ascending=False)
log("\n" + corr_df.round(3).to_string(index=False))
corr_df.to_csv("correlations.csv", index=False)
log("\nSaved: correlations.csv")

# Forest plot
plot_df = corr_df.iloc[::-1].reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7, 6))
for i, row in plot_df.iterrows():
    color = "#2b6cb0" if row["r"] > 0 else "#c53030"
    ax.errorbar(
        row["r"], i,
        xerr=[[row["r"] - row["ci_lo"]], [row["ci_hi"] - row["r"]]],
        fmt="o", color=color, capsize=3, markersize=6,
    )
    if row["p"] < 0.05:
        ax.text(row["ci_hi"] + 0.02, i, "*", va="center", fontsize=14)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df["variable"])
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("Pearson correlation with CGPA (95% CI)")
ax.set_title(f"Bivariate correlations (N = {len(df)})")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig01_correlations.png", dpi=200, bbox_inches="tight")
plt.close()
log("Saved: fig01_correlations.png")




STEP 4 — CORRELATIONS WITH CGPA

                    variable      r  ci_lo  ci_hi     p
        Academic self-rating  0.537  0.430  0.629 0.000
           Daily study hours  0.467  0.350  0.568 0.000
           Daily screen time -0.208 -0.337 -0.071 0.003
            Perceived stress -0.157 -0.290 -0.019 0.026
           Weekly work hours -0.152 -0.285 -0.013 0.032
   Avoid screen before sleep  0.141  0.002  0.275 0.046
               Part-time job -0.120 -0.254  0.019 0.091
                         Age  0.103 -0.036  0.238 0.147
               Phone: gaming -0.089 -0.225  0.051 0.212
            Extracurriculars -0.087 -0.223  0.053 0.221
                 Travel time -0.081 -0.217  0.058 0.254
              Sleep duration  0.076 -0.063  0.213 0.283
        Phone: entertainment -0.066 -0.203  0.074 0.354
             Gender (Male=1) -0.060 -0.197  0.079 0.397
        Phone: study purpose  0.055 -0.085  0.192 0.443
         Financial condition -0.026 -0.164  0.113 0.716
   Travel impa

# STEP 5. STANDARDISE FEATURES FOR CLUSTERING

In [7]:
# =============================================================================
section("STEP 5 — STANDARDISE FEATURES")

CLUSTER_FEATS = [
    "acad_rating", "study_hrs_n", "screen_sleep_b", "Study_Purpose", "trans_n",
    "time_sin", "time_cos", "Entertainment", "fin_cond_n", "sleep_hrs_n",
    "travel_n", "stress_n", "Social_Media", "ECA_b", "work_hrs_n",
]

X = df[CLUSTER_FEATS].values
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
log(f"Feature matrix: shape {Xs.shape} (zero mean, unit variance).")




STEP 5 — STANDARDISE FEATURES
Feature matrix: shape (200, 15) (zero mean, unit variance).


# STEP 6. CHOOSE NUMBER OF CLUSTERS (k)

In [8]:
# =============================================================================
section("STEP 6 — CHOOSE NUMBER OF CLUSTERS")

diag_rows = []
for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=RNG, n_init=10).fit(Xs)
    diag_rows.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": silhouette_score(Xs, km.labels_),
        "calinski_harabasz": calinski_harabasz_score(Xs, km.labels_),
        "davies_bouldin": davies_bouldin_score(Xs, km.labels_),
    })
diag_df = pd.DataFrame(diag_rows)
log("\n" + diag_df.round(3).to_string(index=False))

# Elbow + silhouette plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(diag_df["k"], diag_df["inertia"], "o-", color="#2b6cb0")
axes[0].set(xlabel="k", ylabel="Within-cluster sum of squares", title="Elbow plot")
axes[0].axvline(K, color="red", ls="--", alpha=0.5)
axes[1].plot(diag_df["k"], diag_df["silhouette"], "o-", color="#dd6b20")
axes[1].set(xlabel="k", ylabel="Silhouette coefficient", title="Silhouette")
axes[1].axvline(K, color="red", ls="--", alpha=0.5)
plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig02_k_selection.png", dpi=200, bbox_inches="tight")
plt.close()
log(f"Saved: fig02_k_selection.png  (chose k = {K})")



STEP 6 — CHOOSE NUMBER OF CLUSTERS

 k  inertia  silhouette  calinski_harabasz  davies_bouldin
 2 2690.238       0.098             22.798           2.886
 3 2511.791       0.093             19.145           2.786
 4 2386.429       0.095             16.798           2.530
 5 2307.447       0.084             14.632           2.495
 6 2208.524       0.096             13.905           2.364
 7 2112.563       0.086             13.512           2.322
 8 2082.733       0.075             12.080           2.225
Saved: fig02_k_selection.png  (chose k = 3)


# STEP 7. FIT K-MEANS

In [9]:
# =============================================================================
section("STEP 7 — K-MEANS CLUSTERING")

km = KMeans(n_clusters=K, random_state=RNG, n_init=10).fit(Xs)
df["cluster_raw"] = km.labels_

# Re-label so Cluster 1 = lowest CGPA, Cluster K = highest
order = df.groupby("cluster_raw")["CGPA"].mean().sort_values().index.tolist()
mapping = {old: new + 1 for new, old in enumerate(order)}
df["cluster"] = df["cluster_raw"].map(mapping)

log("Cluster sizes:")
log(df["cluster"].value_counts().sort_index().to_string())



STEP 7 — K-MEANS CLUSTERING
Cluster sizes:
cluster
1    71
2    73
3    56


# STEP 8. PROFILE CLUSTERS

In [10]:
# =============================================================================
section("STEP 8 — CLUSTER PROFILES")

profile_feats = [
    "acad_rating", "study_hrs_n", "sleep_hrs_n", "screen_n", "travel_n",
    "stress_n", "work_hrs_n", "fin_cond_n", "Study_Purpose", "Social_Media",
    "Entertainment", "ECA_b", "screen_sleep_b", "CGPA",
]
profile = df.groupby("cluster")[profile_feats].mean().round(2)
profile["n"] = df["cluster"].value_counts().sort_index()
log("\n" + profile.T.to_string())
profile.to_csv("cluster_profile.csv")
log("\nSaved: cluster_profile.csv")




STEP 8 — CLUSTER PROFILES

cluster             1      2      3
acad_rating      2.24   3.56   3.80
study_hrs_n      1.46   2.27   2.84
sleep_hrs_n      2.18   2.85   2.64
screen_n         3.21   2.55   2.30
travel_n         4.06   3.71   3.84
stress_n         3.85   3.10   3.11
work_hrs_n       5.56   4.06   2.88
fin_cond_n       1.75   1.71   1.50
Study_Purpose    0.41   0.59   0.41
Social_Media     0.52   0.52   0.55
Entertainment    0.58   0.48   0.50
ECA_b            0.56   0.53   0.59
screen_sleep_b   0.56   0.62   0.64
CGPA             3.02   3.29   3.45
n               71.00  73.00  56.00

Saved: cluster_profile.csv


# STEP 9. ANOVA + TUKEY HSD: DO CLUSTERS DIFFER IN CGPA?

In [11]:
# =============================================================================
section("STEP 9 — ANOVA + TUKEY HSD")

groups = [df.loc[df["cluster"] == c, "CGPA"].values for c in sorted(df["cluster"].unique())]
F, p = stats.f_oneway(*groups)
grand_mean = df["CGPA"].mean()
ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
ss_total = float(((df["CGPA"] - grand_mean) ** 2).sum())
eta2 = ss_between / ss_total

log(f"ANOVA: F({K - 1}, {len(df) - K}) = {F:.3f}, p = {p:.4f}, eta^2 = {eta2:.3f}")

tukey = pairwise_tukeyhsd(df["CGPA"], df["cluster"], alpha=0.05)
log("\nTukey HSD pairwise comparisons:")
log(str(tukey))

H, p_kw = stats.kruskal(*groups)
log(f"\nKruskal-Wallis: H = {H:.3f}, p = {p_kw:.4f}")

# Boxplot of CGPA by cluster
palette = {1: "#c53030", 2: "#dd6b20", 3: "#2b6cb0"}
fig, ax = plt.subplots(figsize=(6.5, 4.5))
sns.boxplot(data=df, x="cluster", y="CGPA", hue="cluster",
            palette=palette, ax=ax, legend=False, width=0.5)
sns.stripplot(data=df, x="cluster", y="CGPA", color="black",
              alpha=0.35, size=3, ax=ax)
ax.set(
    xlabel="Cluster (1 = lowest CGPA, 3 = highest)",
    ylabel="CGPA (band midpoint)",
    title=f"CGPA by cluster — F={F:.2f}, p={p:.3f}, eta^2={eta2:.3f}",
)
plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig03_cgpa_by_cluster.png", dpi=200, bbox_inches="tight")
plt.close()
log("Saved: fig03_cgpa_by_cluster.png")



STEP 9 — ANOVA + TUKEY HSD
ANOVA: F(2, 197) = 16.705, p = 0.0000, eta^2 = 0.145

Tukey HSD pairwise comparisons:
Multiple Comparison of Means - Tukey HSD, FWER=0.05
group1 group2 meandiff p-adj   lower  upper  reject
---------------------------------------------------
     1      2   0.2664 0.0005  0.1015 0.4314   True
     1      3   0.4218    0.0  0.2449 0.5987   True
     2      3   0.1553 0.0953 -0.0205 0.3312  False
---------------------------------------------------

Kruskal-Wallis: H = 30.142, p = 0.0000
Saved: fig03_cgpa_by_cluster.png



# STEP 10. PCA VISUALISATION

In [12]:
# =============================================================================
section("STEP 10 — PCA VISUALISATION")

pca = PCA(n_components=2, random_state=RNG)
coords = pca.fit_transform(Xs)
var1, var2 = pca.explained_variance_ratio_[:2] * 100
log(f"Explained variance: PC1 = {var1:.1f}%, PC2 = {var2:.1f}%")

fig, ax = plt.subplots(figsize=(7, 5.5))
for c in sorted(df["cluster"].unique()):
    m = df["cluster"] == c
    ax.scatter(coords[m, 0], coords[m, 1], s=55, alpha=0.75, color=palette[c],
               edgecolor="white", linewidth=0.6,
               label=f"Cluster {c} (n = {m.sum()})")
ax.set(
    xlabel=f"PC1 ({var1:.1f}% variance)",
    ylabel=f"PC2 ({var2:.1f}% variance)",
    title="Student clusters in PCA space",
)
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig04_pca_clusters.png", dpi=200, bbox_inches="tight")
plt.close()
log("Saved: fig04_pca_clusters.png")




STEP 10 — PCA VISUALISATION
Explained variance: PC1 = 13.9%, PC2 = 9.0%
Saved: fig04_pca_clusters.png


# STEP 11. BUILD PREDICTOR MATRIX

In [13]:
# =============================================================================
section("STEP 11 — PREDICTOR MATRIX")

PRED_FEATS = [
    "acad_rating", "study_hrs_n", "sleep_hrs_n", "screen_n", "travel_n",
    "stress_n", "fin_cond_n", "work_hrs_n", "time_sin", "time_cos",
    "Study_Purpose", "Social_Media", "Entertainment", "Gaming",
    "ECA_b", "screen_sleep_b", "pt_job_b", "laptop_b", "travel_impact_b",
    "Gender_b", "Age", "trans_n", "balance_n",
]
Xp = df[PRED_FEATS].values
yp = df["CGPA"].values
log(f"Predictor matrix: {Xp.shape}, target: CGPA")




STEP 11 — PREDICTOR MATRIX
Predictor matrix: (200, 23), target: CGPA


# STEP 12. TRAIN/TEST SPLIT

In [14]:
# =============================================================================
section("STEP 12 — TRAIN/TEST SPLIT")

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    Xp, yp, test_size=TEST_SIZE, random_state=RNG,
)

# Fit scaler on TRAIN only, then apply to test (no leakage)
ml_scaler = StandardScaler().fit(X_train_raw)
X_train = ml_scaler.transform(X_train_raw)
X_test = ml_scaler.transform(X_test_raw)

log(f"Train: {X_train.shape}, Test: {X_test.shape}")
cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RNG)



STEP 12 — TRAIN/TEST SPLIT
Train: (160, 23), Test: (40, 23)


# STEP 13. COMPARE 6 MODELS BY CROSS-VALIDATION

In [15]:
# =============================================================================
section("STEP 13 — COMPARE MODELS (5-fold CV)")

models = {
    "OLS"          : LinearRegression(),
    "Ridge"        : Ridge(alpha=1.0, random_state=RNG),
    "Lasso"        : Lasso(alpha=0.05, random_state=RNG, max_iter=20000),
    "LassoCV"      : LassoCV(cv=cv, random_state=RNG, max_iter=20000),
    "RandomForest" : RandomForestRegressor(n_estimators=500, max_depth=6,
                                           random_state=RNG, n_jobs=-1),
    "GradBoost"    : GradientBoostingRegressor(n_estimators=300, max_depth=3,
                                               learning_rate=0.05,
                                               random_state=RNG),
}

cv_rows = []
for name, model in models.items():
    r2 = cross_val_score(model, X_train, y_train, cv=cv, scoring="r2")
    mae = -cross_val_score(model, X_train, y_train, cv=cv,
                           scoring="neg_mean_absolute_error")
    cv_rows.append({
        "model": name,
        "r2_mean": r2.mean(), "r2_sd": r2.std(),
        "mae_mean": mae.mean(), "mae_sd": mae.std(),
    })

cv_df = pd.DataFrame(cv_rows).sort_values("r2_mean", ascending=False)
log("\n" + cv_df.round(3).to_string(index=False))

best_name = cv_df.iloc[0]["model"]
log(f"\nBest model: {best_name}")



STEP 13 — COMPARE MODELS (5-fold CV)

       model  r2_mean  r2_sd  mae_mean  mae_sd
     LassoCV    0.375  0.077     0.284   0.029
       Lasso    0.373  0.076     0.284   0.028
       Ridge    0.272  0.058     0.297   0.024
         OLS    0.268  0.059     0.298   0.024
RandomForest    0.254  0.105     0.316   0.022
   GradBoost    0.137  0.127     0.323   0.019

Best model: LassoCV


# STEP 14. HYPERPARAMETER TUNING

In [16]:
# =============================================================================
section("STEP 14 — HYPERPARAMETER TUNING")

PARAM_GRIDS = {
    "Lasso":        ({"alpha": [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]},
                     Lasso(random_state=RNG, max_iter=20000)),
    "LassoCV":      ({"alpha": [0.001, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2]},
                     Lasso(random_state=RNG, max_iter=20000)),
    "Ridge":        ({"alpha": [0.01, 0.1, 1.0, 10, 100]},
                     Ridge(random_state=RNG)),
    "RandomForest": ({"n_estimators": [300, 500, 800],
                      "max_depth": [3, 5, 7, None],
                      "min_samples_leaf": [1, 2, 5]},
                     RandomForestRegressor(random_state=RNG, n_jobs=-1)),
    "GradBoost":    ({"n_estimators": [200, 400, 600],
                      "max_depth": [2, 3, 4],
                      "learning_rate": [0.02, 0.05, 0.1]},
                     GradientBoostingRegressor(random_state=RNG)),
}

if best_name in PARAM_GRIDS:
    grid, base = PARAM_GRIDS[best_name]
    gs = GridSearchCV(base, grid, cv=cv, scoring="r2", n_jobs=-1)
    gs.fit(X_train, y_train)
    log(f"Best params: {gs.best_params_}")
    log(f"Best CV R^2: {gs.best_score_:.3f}")
    final_model = gs.best_estimator_
else:
    final_model = models[best_name].fit(X_train, y_train)
    log("No tunable hyperparameters; using default.")




STEP 14 — HYPERPARAMETER TUNING
Best params: {'alpha': 0.05}
Best CV R^2: 0.373


# STEP 15. EVALUATE ON HELD-OUT TEST SET

In [17]:
# =============================================================================
section("STEP 15 — TEST-SET EVALUATION")

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

test_r2 = r2_score(y_test, y_pred)
test_mae = mean_absolute_error(y_test, y_pred)
test_rmse = float(np.sqrt(np.mean((y_test - y_pred) ** 2)))

log(f"Test R^2  = {test_r2:.3f}")
log(f"Test MAE  = {test_mae:.3f} CGPA points")
log(f"Test RMSE = {test_rmse:.3f} CGPA points")

# Predicted vs actual plot
fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(y_test, y_pred, s=55, alpha=0.75, color="#2b6cb0",
           edgecolor="white", linewidth=0.6)
lo = float(min(y_test.min(), y_pred.min())) - 0.2
hi = float(max(y_test.max(), y_pred.max())) + 0.2
ax.plot([lo, hi], [lo, hi], "r--", alpha=0.6, label="y = x")
ax.set(
    xlabel="Actual CGPA", ylabel="Predicted CGPA",
    xlim=(lo, hi), ylim=(lo, hi),
    title=f"Predicted vs actual CGPA (test R^2 = {test_r2:.2f})",
)
ax.legend()
plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig05_pred_vs_actual.png", dpi=200, bbox_inches="tight")
plt.close()
log("Saved: fig05_pred_vs_actual.png")



STEP 15 — TEST-SET EVALUATION
Test R^2  = 0.083
Test MAE  = 0.359 CGPA points
Test RMSE = 0.448 CGPA points
Saved: fig05_pred_vs_actual.png


# STEP 16. FEATURE IMPORTANCE

In [18]:
# =============================================================================
section("STEP 16 — FEATURE IMPORTANCE")

# Random Forest impurity-based importance
rf = RandomForestRegressor(n_estimators=500, max_depth=6,
                           random_state=RNG, n_jobs=-1).fit(X_train, y_train)
rf_imp = pd.Series(rf.feature_importances_, index=PRED_FEATS).sort_values()

fig, ax = plt.subplots(figsize=(7, 7))
rf_imp.plot(kind="barh", ax=ax, color="#2b6cb0")
ax.set(xlabel="Importance", title="Random Forest feature importance")
plt.tight_layout()
plt.savefig(f"{OUTDIR}/fig06_feature_importance.png", dpi=200, bbox_inches="tight")
plt.close()
log("Saved: fig06_feature_importance.png")

log("\nTop 10 by Random Forest importance:")
log(rf_imp.iloc[::-1].head(10).round(3).to_string())

# LassoCV non-zero coefficients (signed)
lcv = LassoCV(cv=cv, random_state=RNG, max_iter=20000).fit(X_train, y_train)
coef = pd.Series(lcv.coef_, index=PRED_FEATS)
nonzero = coef[coef != 0].sort_values(key=abs, ascending=False)

log(f"\nLassoCV optimal alpha: {lcv.alpha_:.4f}")
log(f"Non-zero coefficients ({len(nonzero)} of {len(coef)}):")
log(nonzero.round(3).to_string())




STEP 16 — FEATURE IMPORTANCE
Saved: fig06_feature_importance.png

Top 10 by Random Forest importance:
acad_rating    0.365
study_hrs_n    0.099
Age            0.084
screen_n       0.047
travel_n       0.046
stress_n       0.041
sleep_hrs_n    0.040
work_hrs_n     0.031
trans_n        0.025
fin_cond_n     0.025

LassoCV optimal alpha: 0.0393
Non-zero coefficients (6 of 23):
acad_rating       0.169
study_hrs_n       0.103
laptop_b         -0.022
screen_sleep_b    0.008
ECA_b            -0.006
Gender_b         -0.001


# STEP 17. SAVE OUTPUTS

In [19]:
# =============================================================================
section("STEP 17 — SAVE OUTPUTS")

# Dataset with cluster labels
out_cols = [
    "Age", "Gender", "CGPA?", "CGPA", "study_hrs", "study_time", "acad_rating",
    "fin_cond", "pt_job", "work_hrs", "screen_time", "phone_use", "travel_time",
    "trans_mode", "sleep_hrs", "ECA", "stress", "screen_sleep", "balance_diff",
    "cluster",
]
df[out_cols].to_csv("dataset_with_clusters.csv", index=False)
log("Saved: dataset_with_clusters.csv")

# Headline numbers as JSON
results = {
    "n": int(len(df)),
    "anova": {"F": float(F), "p": float(p), "eta_squared": float(eta2)},
    "kruskal": {"H": float(H), "p": float(p_kw)},
    "cluster_means": {int(c): float(df.loc[df["cluster"] == c, "CGPA"].mean())
                      for c in sorted(df["cluster"].unique())},
    "cluster_sizes": {int(c): int((df["cluster"] == c).sum())
                      for c in sorted(df["cluster"].unique())},
    "best_model": str(best_name),
    "test_r2": float(test_r2),
    "test_mae": float(test_mae),
    "test_rmse": float(test_rmse),
    "lasso_alpha": float(lcv.alpha_),
    "lasso_nonzero_features": list(nonzero.index),
    "pca_explained_variance": [float(var1) / 100, float(var2) / 100],
}
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)
log("Saved: results.json")

# Full text log
with open("results_summary.txt", "w") as f:
    f.write("\n".join(LOG))
log("Saved: results_summary.txt")

log("\nPIPELINE COMPLETE")
log(f"\nFigures in: {OUTDIR}/")
log("  fig01_correlations.png       Bivariate correlations forest plot")
log("  fig02_k_selection.png        Elbow + silhouette diagnostics")
log("  fig03_cgpa_by_cluster.png    CGPA boxplot by cluster")
log("  fig04_pca_clusters.png       PCA scatter coloured by cluster")
log("  fig05_pred_vs_actual.png     Test-set predictions")
log("  fig06_feature_importance.png Random Forest feature importance")

# Re-write log at the end so it includes the final lines
with open("results_summary.txt", "w") as f:
    f.write("\n".join(LOG))


STEP 17 — SAVE OUTPUTS
Saved: dataset_with_clusters.csv
Saved: results.json
Saved: results_summary.txt

PIPELINE COMPLETE

Figures in: figs/
  fig01_correlations.png       Bivariate correlations forest plot
  fig02_k_selection.png        Elbow + silhouette diagnostics
  fig03_cgpa_by_cluster.png    CGPA boxplot by cluster
  fig04_pca_clusters.png       PCA scatter coloured by cluster
  fig05_pred_vs_actual.png     Test-set predictions
  fig06_feature_importance.png Random Forest feature importance
